# Some nGram Analysis of the FAIR-CARE Survey Text Responses

In [1]:

from IPython.display import display

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency


processed_survey_path = '/home/ekansa/oc-data/fair-care-survey-processed.csv' # Keep this OUT of version control, has sensitive info
wide_survey_path = '/home/ekansa/oc-data/fair-care-survey-multi-select-expanded.csv' # Keep this OUT of version control, has sensitive info
ngram_freq_path = '/home/ekansa/oc-data/fair-care-ngram-freq.csv'
ngram_chi_tests_path = '/home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv'
ngram_chi_significant_path = '/home/ekansa/oc-data/fair-care-ngram-freq-chi-tests-sig.csv'
ngram_chi_interesting_significant_path = '/home/ekansa/oc-data/fair-care-ngram-freq-chi-tests-interesting-sig.csv'

df = pd.read_csv(processed_survey_path, low_memory=False)
df_ngram = pd.read_csv(ngram_freq_path, low_memory=False)
df_wide = pd.read_csv(wide_survey_path, low_memory=False)

# An Ngram needs to have a count of this or higher to be
# considered for the chi-square significance tests
MIN_NGRAM_COUNT = 4

print(f'FAIR+CARE survey has {len(df.index)} rows')
print(f'FAIR+CARE survey wide_survey data has {len(df_wide.index)} rows, {len(df_wide.columns)} columns')
print(f'FAIR+CARE survey ngram extract has {len(df_ngram.index)} rows')

FAIR+CARE survey has 787 rows
FAIR+CARE survey wide_survey data has 787 rows, 271 columns
FAIR+CARE survey ngram extract has 216860 rows


In [2]:
gen_group_cols = ['column', 'ngram_len', 'ngram',]
df_ngram_gp1 = df_ngram[(gen_group_cols + ['ngram_count',])].groupby(gen_group_cols, as_index=False).sum()
df_ngram_gp1.sort_values(by=['column', 'ngram_count',], ascending=[True, False], inplace=True)
print(f'There are {len(df_ngram_gp1.index)} unique ngrams')

# Repeatly used ngrams
rep_ngram_index = df_ngram_gp1['ngram_count'] >= 10
print(f'There are {len(df_ngram_gp1[rep_ngram_index].index)} unique ngrams used 10 or more times')

df_ngram_gp1[rep_ngram_index].head(100)

There are 207627 unique ngrams
There are 80 unique ngrams used 10 or more times


,column,ngram_len,ngram,ngram_count
248,"Accessible: Additional info, Text",2,data is,13
7531,Application of Ethical Frameworks: Text,2,not sure,20
8073,Application of Ethical Frameworks: Text,2,t know,17
9282,Application of Ethical Frameworks: Text,3,don t know,17
8370,Application of Ethical Frameworks: Text,2,try to,14
...,...,...,...,...
177523,"Response Type: Org, Text",2,academic unit,11
194919,Survey Comments: Text,2,the survey,15
194904,Survey Comments: Text,2,the questions,13
194973,Survey Comments: Text,2,this survey,11


In [3]:
gen_group_cols = ['Section', 'ngram_len', 'ngram',]
df_ngram_gp2 = df_ngram[(gen_group_cols + ['ngram_count',])].groupby(gen_group_cols, as_index=False).sum()
df_ngram_gp2.sort_values(by=['Section', 'ngram_count',], ascending=[True, False], inplace=True)
print(f'There are {len(df_ngram_gp1.index)} unique ngrams')

There are 207627 unique ngrams


In [4]:
# Repeatly used ngrams
# First Show results for the CARE
ngram_count_threshold = 10
section = 'CARE'
rep_ngram_index = ((df_ngram_gp2['ngram_count'] >= ngram_count_threshold) & (df_ngram_gp2['ngram_len'] >= 3) & df_ngram_gp2['Section'].isin([section]))
print(f'There are {len(df_ngram_gp2[rep_ngram_index].index)} unique ngrams used {ngram_count_threshold} or more times with {section} responses')

df_ngram_gp2[rep_ngram_index].head(10)

There are 106 unique ngrams used 10 or more times with CARE responses


,Section,ngram_len,ngram,ngram_count
29955,CARE,3,don t know,105
66100,CARE,4,i don t know,87
40494,CARE,3,or descendant communities,45
34868,CARE,3,indigenous peoples and,42
26610,CARE,3,case by case,39
26255,CARE,3,by case basis,30
58926,CARE,4,case by case basis,30
49129,CARE,3,to indigenous peoples,29
21193,CARE,3,a case by,27
23425,CARE,3,and or descendant,27


In [5]:
# Repeatly used ngrams
# First Show results for the CARE
ngram_count_threshold = 10
section = 'FAIR'
rep_ngram_index = ((df_ngram_gp2['ngram_count'] >= ngram_count_threshold) & (df_ngram_gp2['ngram_len'] >= 3) & df_ngram_gp2['Section'].isin([section]))
print(f'There are {len(df_ngram_gp2[rep_ngram_index].index)} unique ngrams used {ngram_count_threshold} or more times with {section} responses')

df_ngram_gp2[rep_ngram_index].head(10)

There are 4 unique ngrams used 10 or more times with FAIR responses


,Section,ngram_len,ngram,ngram_count
137325,FAIR,3,of the data,16
141210,FAIR,3,to the public,12
133982,FAIR,3,depends on the,11
140336,FAIR,3,the data is,11


In [6]:
def make_filter_indices(df_wc):
    """Make lists of filter indices for charts"""
    crm_cols = [
        'Work: Work setting::CRM: Cultural Resources Consulting Firm::TF',
        'Work: Work setting::CRM: Tribally-Owned or similar Consulting Firm::TF',
        'Work: Work setting::CRM: Museum or University-based Consulting Organization::TF',
        'Work: Work setting::Construction Management Firm::TF',
        'Work: Work setting::CRM: Environmental or Engineering Consulting Firm::TF',
    ]

    # Make an index to select responses that indicate work in ANY CRM setting (lots of 'or' selections)
    any_crm_index = (df_wc[crm_cols[0]] == True)
    for crm_col in crm_cols[1:]:
        # Add an "or" option for the next CRM column
        any_crm_index |= (df_wc[crm_col] == True)
    
    
    work_group_configs = [
        ((df_wc['Work: Work setting::Academic: University or College::TF'] == True), '\nAcademic (Univ)', '-academic',),
        ((df_wc['Work: Work setting::Academic: University or College::TF'] == False), '\nNot Academic (Univ)', '-not-academic',),
        (any_crm_index, '\nCRM', '-crm',),
    ]
    
    region_configs = [
        ((df_wc['Region Focus: Work/research Region::North America (specify region)::TF'] == True), '\nN. America Focus', '-n-america',),
        ((df_wc['Region Focus: Work/research Region::North America (specify region)::TF'] == False), '\nOutside N. America Focus', '-out-n-america',),
    ]
    
    org_configs = [
        ((df_wc['Response Type: Individual or Org '] == 'Individual'), '\nResponding as Individual', '-resp-ind',),
        ((df_wc['Response Type: Individual or Org '].str.contains('Organization')), '\nResponding as Organization', '-resp-org',),
    ]
    
    filter_segment_configs = [
        # (filter_index, title_suffix, file_suffix,),
        ((~df_wc['Response ID'].isnull()), '\nAll Reponses (Unfiltered)', '-all',),
    ] + work_group_configs + region_configs + org_configs

    return work_group_configs, region_configs, org_configs, filter_segment_configs


In [7]:
# Columns used to identify responses from CRM settings
crm_cols = [
    'Work: Work setting::CRM: Cultural Resources Consulting Firm::TF',
    'Work: Work setting::CRM: Tribally-Owned or similar Consulting Firm::TF',
    'Work: Work setting::CRM: Museum or University-based Consulting Organization::TF',
    'Work: Work setting::Construction Management Firm::TF',
    'Work: Work setting::CRM: Environmental or Engineering Consulting Firm::TF',
]

wide_cols = [
    'Response ID', 
    'Demo: Age Range',	
    'Demo: Gender', 
    'Work: Work setting::Academic: University or College::TF',
    'Region Focus: Work/research Region::North America (specify region)::TF',
    'Response Type: Individual or Org ',
] + crm_cols

# Merge certain columns from the df_wide into the df_token dataset so we can
# filter by criteria from the df_wide.
df_wc = pd.merge(df_ngram, df_wide[wide_cols], how='left', on=['Response ID'])

In [8]:
_, _, _, filter_segment_configs = make_filter_indices(df_wc)

df_agg = None
gen_group_cols = ['Section', 'Sub-Section', 'ngram_len', 'ngram',]
segments = []
for filter_index, title_suffix, file_suffix in filter_segment_configs:
    filter_index &= ~df_wc['Response ID'].isnull()
    segment = title_suffix.replace('\n', '')
    if segment != 'All Reponses (Unfiltered)':
        segments.append(segment)
    segment_size = df_wc[filter_index]['Response ID'].nunique()
    df_act_grp = df_wc[filter_index][(gen_group_cols + ['ngram_count',])].groupby(gen_group_cols, as_index=False).agg(
        {
            'ngram_count': 'sum',
        }
    )
    segment_count_col = f'ngram_count_{segment}'
    df_act_grp.rename(columns={'ngram_count': segment_count_col}, inplace=True)
    df_act_grp[[segment_count_col]] = df_act_grp[[segment_count_col]].fillna(value=0)
    df_act_grp[f'segment_size_{segment}'] = int(segment_size)
    if df_agg is None:
        df_agg = df_act_grp.copy()
    else:
        df_agg = pd.merge(df_agg, df_act_grp, how='outer', on=gen_group_cols)

df_agg.head(10)

,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),segment_size_All Reponses (Unfiltered),ngram_count_Academic (Univ),segment_size_Academic (Univ),ngram_count_Not Academic (Univ),segment_size_Not Academic (Univ),ngram_count_CRM,segment_size_CRM,ngram_count_N. America Focus,segment_size_N. America Focus,ngram_count_Outside N. America Focus,segment_size_Outside N. America Focus,ngram_count_Responding as Individual,segment_size_Responding as Individual,ngram_count_Responding as Organization,segment_size_Responding as Organization
0,CARE,Authority to Control,2,100 access,1,576,NaN,NaN,1.0,398.0,NaN,NaN,1.0,407.0,NaN,NaN,NaN,NaN,1.0,190.0
1,CARE,Authority to Control,2,100 of,1,576,1.0,172.0,NaN,NaN,NaN,NaN,1.0,407.0,NaN,NaN,1.0,371.0,NaN,NaN
2,CARE,Authority to Control,2,106 laws,1,576,1.0,172.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,149.0,1.0,371.0,NaN,NaN
3,CARE,Authority to Control,2,20 years,1,576,1.0,172.0,NaN,NaN,NaN,NaN,1.0,407.0,NaN,NaN,1.0,371.0,NaN,NaN
4,CARE,Authority to Control,2,20th century,2,576,2.0,172.0,NaN,NaN,NaN,NaN,1.0,407.0,1.0,149.0,1.0,371.0,1.0,190.0
5,CARE,Authority to Control,2,30 years,1,576,1.0,172.0,NaN,NaN,NaN,NaN,1.0,407.0,NaN,NaN,1.0,371.0,NaN,NaN
6,CARE,Authority to Control,2,33 above,1,576,1.0,172.0,NaN,NaN,NaN,NaN,1.0,407.0,NaN,NaN,NaN,NaN,1.0,190.0
7,CARE,Authority to Control,2,5 over,1,576,NaN,NaN,1.0,398.0,1.0,141.0,1.0,407.0,NaN,NaN,1.0,371.0,NaN,NaN
8,CARE,Authority to Control,2,50 years,1,576,1.0,172.0,NaN,NaN,1.0,141.0,1.0,407.0,NaN,NaN,1.0,371.0,NaN,NaN
9,CARE,Authority to Control,2,_not_ a,1,576,NaN,NaN,1.0,398.0,NaN,NaN,1.0,407.0,NaN,NaN,1.0,371.0,NaN,NaN


In [9]:
sum_cols = ['Section', 'Sub-Section', 'ngram_len',]
total_col = 'ngram_count_All Reponses (Unfiltered)'

In [10]:
# Calculate rates for all of the ngrams in each segment, sub-segment
rate_segments = ['All Reponses (Unfiltered)'] + segments
for segment in rate_segments:
    segment_rate_col = f'ngram_rate_{segment}'
    segment_rate_v_all_col = f'ngram_rate_over_all::{segment}'
    df_agg[segment_rate_col] = np.nan
    df_agg[segment_rate_v_all_col] = np.nan

for section in df_agg['Section'].unique().tolist():
    section_index = (df_agg['Section'] == section)
    for sub_section in df_agg[section_index]['Sub-Section'].unique().tolist():
        sect_sub_index = (df_agg['Sub-Section'] == sub_section) & section_index
        for ngram_len in df_agg[sect_sub_index]['ngram_len'].unique().tolist():
            act_index = (df_agg['ngram_len'] == ngram_len) & sect_sub_index
            for segment in rate_segments:
                segment_count_col = f'ngram_count_{segment}'
                segment_rate_col = f'ngram_rate_{segment}'
                segment_total_count = df_agg[act_index][segment_count_col].sum()
                df_agg.loc[act_index, segment_rate_col] = df_agg[act_index][segment_count_col] / segment_total_count

for segment in segments:
    segment_rate_col = f'ngram_rate_{segment}'
    segment_rate_v_all_col = f'ngram_rate_over_all::{segment}'
    df_agg[segment_rate_v_all_col] = (
        (df_agg[segment_rate_col] - df_agg['ngram_rate_All Reponses (Unfiltered)'])
        / ((df_agg[segment_rate_col] + df_agg['ngram_rate_All Reponses (Unfiltered)']) * 0.5)
    ) * 100
    

In [11]:
p_val_cols = []
for segment in segments:
    segment_p_value_col = f'chi_p_value::{segment}'
    segment_count_col = f'ngram_count_{segment}'
    segment_rate_col = f'ngram_rate_{segment}'
    segment_rate_v_all_col = f'ngram_rate_over_all::{segment}'
    p_val_cols.append(segment_p_value_col)
    df_agg[segment_p_value_col] = np.nan
    df_agg[[segment_rate_col]] = df_agg[[segment_rate_col]].fillna(value=0)
    segment_count_index = (df_agg[total_col] >= MIN_NGRAM_COUNT)
    for _, row in df_agg[segment_count_index].iterrows():
        ngram = row['ngram']
        ngram_len = row['ngram_len']
        section = row['Section']
        sub_section = row['Sub-Section']
        section_index = (
            (df_agg['ngram_len'] == ngram_len)
            & (df_agg['Section'] == section)
            & (df_agg['Sub-Section'] == sub_section)
        )
        all_section_count_index =  ~(df_agg['ngram'] == ngram) & section_index
        contingency_table = [
            [row[segment_count_col], df_agg[all_section_count_index][segment_count_col].sum()], 
            [row[total_col], df_agg[all_section_count_index][total_col].sum(),]
        ]
        chi2, p_value, _, _ = chi2_contingency(contingency_table)
        p_value_index = (df_agg['ngram'] == ngram) & section_index
        df_agg.loc[p_value_index, segment_p_value_col] = p_value
    print('\n')
    print('-'*50)
    print(f'Completed {segment}. Saving results to {ngram_chi_tests_path}')
    df_agg.to_csv(ngram_chi_tests_path, index=False)
    segment_p_value_index = ~df_agg[segment_p_value_col].isnull() & (df_agg[segment_p_value_col] < 0.05)
    print(f'Interesting ngrams for {segment}')
    display_cols = sum_cols + ['ngram', total_col, segment_count_col, segment_rate_v_all_col] 
    display(df_agg[segment_p_value_index][display_cols].head(20))
        




--------------------------------------------------
Completed Academic (Univ). Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for Academic (Univ)


,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_Academic (Univ),ngram_rate_over_all::Academic (Univ)
3187,CARE,Authority to Control,2,my research,13,11.0,93.742228
3351,CARE,Authority to Control,2,not applicable,13,11.0,93.742228
6033,CARE,Authority to Control,2,work with,20,13.0,71.942026
32413,CARE,Collective Benefit,2,in place,13,9.0,91.157821
55446,CARE,Ethics,2,my research,8,8.0,102.996416
92831,CARE,General,2,with indigenous,15,12.0,85.420090
130941,FAIR,Accessible,2,data is,15,8.0,98.955447
131631,FAIR,Accessible,2,publicly accessible,8,6.0,122.489652
140458,FAIR,Findable,2,not yet,11,9.0,95.697090
169357,FAIR,Reusable,2,google docs,5,4.0,128.995798




--------------------------------------------------
Completed Not Academic (Univ). Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for Not Academic (Univ)


,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_Not Academic (Univ),ngram_rate_over_all::Not Academic (Univ)




--------------------------------------------------
Completed CRM. Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for CRM


...

,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_Outside N. America Focus,ngram_rate_over_all::Outside N. America Focus
2431,CARE,Authority to Control,2,i work,18,10.0,86.681178
3187,CARE,Authority to Control,2,my research,13,10.0,111.168126
3351,CARE,Authority to Control,2,not applicable,13,10.0,111.168126
4950,CARE,Authority to Control,2,t work,6,6.0,127.976654
6021,CARE,Authority to Control,2,work in,14,9.0,98.151805
8387,CARE,Authority to Control,3,don t work,6,6.0,127.948509
9334,CARE,Authority to Control,3,i work in,6,5.0,116.544748
12505,CARE,Authority to Control,3,t work with,6,6.0,127.948509
13982,CARE,Authority to Control,3,with indigenous people,7,6.0,118.389978
14113,CARE,Authority to Control,3,work with indigenous,11,7.0,97.342805




--------------------------------------------------
Completed Responding as Individual. Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for Responding as Individual


,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_Responding as Individual,ngram_rate_over_all::Responding as Individual




--------------------------------------------------
Completed Responding as Organization. Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for Responding as Organization


,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_Responding as Organization,ngram_rate_over_all::Responding as Organization
107199,CARE,Responsibility,2,yes i,30,3.0,-121.288252
174114,General,General,2,1 5,17,17.0,81.061628
174123,General,General,2,11 20,16,16.0,81.061628
174159,General,General,2,6 10,13,13.0,81.061628


In [12]:
segment_comps = [
    ('Academic (Univ)', 'Not Academic (Univ)',),
    ('CRM', 'Academic (Univ)',),
    ('Outside N. America Focus', 'N. America Focus', ),
    ('Responding as Organization', 'Responding as Individual'),
]
for segment, comp_segment in segment_comps:
    segment_comp_p_value_col = f'chi_p_value::{segment}::vs::{comp_segment}'
    segment_count_col = f'ngram_count_{segment}'
    segment_rate_col = f'ngram_rate_{segment}'
    segment_rate_v_all_col = f'ngram_rate_over_all::{segment}'
    segment_comp_count_col = f'ngram_count_{comp_segment}'
    segment_comp_rate_v_all_col = f'ngram_rate_over_all::{comp_segment}'
    
    p_val_cols.append(segment_comp_p_value_col)
    df_agg[segment_comp_p_value_col] = np.nan
    segment_count_index = (df_agg[total_col] > 4)
    for _, row in df_agg[segment_count_index].iterrows():
        if row[segment_count_col] == 0 and row[segment_comp_count_col] == 0:
            continue
        ngram = row['ngram']
        ngram_len = row['ngram_len']
        section = row['Section']
        sub_section = row['Sub-Section']
        section_index = (
            (df_agg['ngram_len'] == ngram_len)
            & (df_agg['Section'] == section)
            & (df_agg['Sub-Section'] == sub_section)
        )
        all_section_count_index =  ~(df_agg['ngram'] == ngram) & section_index
        contingency_table = [
            [row[segment_count_col], df_agg[all_section_count_index][segment_count_col].sum()], 
            [row[segment_comp_count_col], df_agg[all_section_count_index][segment_comp_count_col].sum(),]
        ]
        chi2, p_value, _, _ = chi2_contingency(contingency_table)
        p_value_index = (df_agg['ngram'] == ngram) & section_index
        df_agg.loc[p_value_index, segment_comp_p_value_col] = p_value
    print('\n')
    print('-'*50)
    print(f'Completed {segment} versus {comp_segment}. Saving results to {ngram_chi_tests_path}')
    df_agg.to_csv(ngram_chi_tests_path, index=False)
    segment_p_value_index = ~df_agg[segment_comp_p_value_col].isnull() & (df_agg[segment_comp_p_value_col] < 0.05)
    print(f'Interesting ngrams for {segment} versus {comp_segment}')
    display_cols = sum_cols + ['ngram', total_col, segment_count_col, segment_rate_v_all_col, segment_comp_count_col, segment_comp_rate_v_all_col] 
    display(df_agg[segment_p_value_index][display_cols].head(20))





--------------------------------------------------
Completed Academic (Univ) versus Not Academic (Univ). Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for Academic (Univ) versus Not Academic (Univ)


,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_Academic (Univ),ngram_rate_over_all::Academic (Univ),ngram_count_Not Academic (Univ),ngram_rate_over_all::Not Academic (Univ)
2431,CARE,Authority to Control,2,i work,18,12.0,74.136109,6.0,-69.676472
3187,CARE,Authority to Control,2,my research,13,11.0,93.742228,2.0,-127.053224
3341,CARE,Authority to Control,2,none of,8,7.0,96.337406,1.0,-138.632351
3351,CARE,Authority to Control,2,not applicable,13,11.0,93.742228,2.0,-127.053224
4347,CARE,Authority to Control,2,related to,9,6.0,74.136109,3.0,-69.676472
5349,CARE,Authority to Control,2,this point,6,5.0,92.546638,1.0,-122.157320
5481,CARE,Authority to Control,2,to indigenous,13,8.0,67.131482,5.0,-56.804301
5959,CARE,Authority to Control,2,with indigenous,20,11.0,56.983488,8.0,-53.178889
6021,CARE,Authority to Control,2,work in,14,9.0,70.978112,4.0,-82.841059
6033,CARE,Authority to Control,2,work with,20,13.0,71.942026,7.0,-65.353742




--------------------------------------------------
Completed CRM versus Academic (Univ). Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for CRM versus Academic (Univ)


,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_CRM,ngram_rate_over_all::CRM,ngram_count_Academic (Univ),ngram_rate_over_all::Academic (Univ)
4947,CARE,Authority to Control,2,t think,9,6.0,100.928240,1.0,-93.468988
5856,CARE,Authority to Control,2,we follow,10,6.0,92.868518,1.0,-101.498729
5959,CARE,Authority to Control,2,with indigenous,20,1.0,-125.783090,11.0,56.983488
6023,CARE,Authority to Control,2,work is,10,7.0,104.519825,1.0,-101.498729
6033,CARE,Authority to Control,2,work with,20,2.0,-74.796688,13.0,71.942026
8384,CARE,Authority to Control,3,don t think,9,6.0,101.454106,1.0,-93.771840
17479,CARE,Authority to Control,4,i don t think,9,6.0,102.399245,1.0,-93.535616
34200,CARE,Collective Benefit,2,the data,20,10.0,87.428273,2.0,-88.517956
55629,CARE,Ethics,2,not sure,23,10.0,70.440775,3.0,-84.207835
129734,Demographics,General,2,united states,29,12.0,61.590543,3.0,-86.045837




--------------------------------------------------
Completed Outside N. America Focus versus N. America Focus. Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for Outside N. America Focus versus N. America Focus


,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_Outside N. America Focus,ngram_rate_over_all::Outside N. America Focus,ngram_count_N. America Focus,ngram_rate_over_all::N. America Focus
2431,CARE,Authority to Control,2,i work,18,10.0,86.681178,8.0,-53.980078
2595,CARE,Authority to Control,2,indigenous people,14,7.0,77.932569,7.0,-42.896417
3187,CARE,Authority to Control,2,my research,13,10.0,111.168126,3.0,-108.043093
3197,CARE,Authority to Control,2,n a,8,5.0,95.998455,3.0,-69.343151
3351,CARE,Authority to Control,2,not applicable,13,10.0,111.168126,3.0,-108.043093
5959,CARE,Authority to Control,2,with indigenous,20,10.0,77.932569,10.0,-42.896417
6021,CARE,Authority to Control,2,work in,14,9.0,98.151805,5.0,-73.598812
6100,CARE,Authority to Control,2,yes as,6,4.0,100.887957,2.0,-79.486721
9334,CARE,Authority to Control,3,i work in,6,5.0,116.544748,1.0,-128.994269
10578,CARE,Authority to Control,3,none of my,5,4.0,113.816660,1.0,-117.714494




--------------------------------------------------
Completed Responding as Organization versus Responding as Individual. Saving results to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests.csv
Interesting ngrams for Responding as Organization versus Responding as Individual


,Section,Sub-Section,ngram_len,ngram,ngram_count_All Reponses (Unfiltered),ngram_count_Responding as Organization,ngram_rate_over_all::Responding as Organization,ngram_count_Responding as Individual,ngram_rate_over_all::Responding as Individual
3187,CARE,Authority to Control,2,my research,13,1.0,-136.733468,12.0,45.898770
4059,CARE,Authority to Control,2,prior to,7,6.0,70.698907,1.0,-120.781937
4941,CARE,Authority to Control,2,t know,13,1.0,-136.733468,12.0,45.898770
5596,CARE,Authority to Control,2,tribal lands,7,6.0,70.698907,1.0,-120.781937
6162,CARE,Authority to Control,2,yes we,12,10.0,68.221667,2.0,-110.532042
8379,CARE,Authority to Control,3,don t know,13,1.0,-136.126560,12.0,45.010470
92455,CARE,General,2,t know,63,15.0,-41.454510,48.0,19.238941
93778,CARE,General,3,don t know,63,15.0,-41.719544,48.0,19.298068
97118,CARE,General,4,i don t know,58,13.0,-48.158681,45.0,21.440401
107199,CARE,Responsibility,2,yes i,30,3.0,-121.288252,27.0,43.149917


In [13]:
index_any_sig_p_value = (df_agg[p_val_cols[0]] < 0.05)
for p_val_col in p_val_cols[1:]:
    index_any_sig_p_value |= (df_agg[p_val_col] < 0.05)

seg_cols = []
for segment in segments:
    for col in df_agg.columns.tolist():
        if col in seg_cols:
            continue
        if '::vs::' in col:
            continue
        if not col.endswith(segment):
            continue
        match_col = None
        for start in ['ngram_rate_over_all::', 'ngram_count_', 'chi_p_value_']:
            if col.startswith(start):
                match_col = start + segment
        if col == match_col:
            seg_cols.append(col)

fewer_cols = []
more_cols = []
for segment in segments:
    fewer_col = f'fewer_{segment}'
    fewer_cols.append(fewer_col)
    more_col = f'more_{segment}'
    more_cols.append(more_col)
    df_agg[fewer_col] = ''
    df_agg[more_col] = ''
    p_val_col = f'chi_p_value_{segment}'
    seg_rate_col = f'ngram_rate_over_all::{segment}'
    segment_pval_cols = [c for c in df_agg.columns.tolist() if c.startswith('chi_p_value') and f'::{segment}' in c]
    seg_sig_index = (df_agg[segment_pval_cols[0]] < 0.05)
    for p_val_col in segment_pval_cols[1:]:
        seg_sig_index |= (df_agg[p_val_col] < 0.05)
    for p_val_col in p_val_cols[1:]:
        index_any_sig_p_value |= (df_agg[p_val_col] < 0.05)
    fewer_index = (df_agg[seg_rate_col] < 0) & seg_sig_index
    more_index = (df_agg[seg_rate_col] > 0) & seg_sig_index
    df_agg.loc[fewer_index, fewer_col] = 'yes'
    df_agg.loc[more_index,  more_col] = 'yes'
    
first_cols = sum_cols + fewer_cols + ['ngram', total_col,] + more_cols
vs_cols = [col for col in df_agg.columns.tolist() if '::vs::' in col]

df_sig = df_agg[index_any_sig_p_value][(first_cols + seg_cols + vs_cols)].copy()
df_sig.sort_values(by=['Section', 'Sub-Section', 'ngram_count_All Reponses (Unfiltered)', 'ngram'], ascending=[True, True, False, True], inplace=True)
print(f'There are {len(df_sig.index)} significant ngrams')




There are 176 significant ngrams


In [14]:
# Drop lower length ngrams if they are exactly in longer ngrams
drops = []
for check_length in [2,3,4,]:
    len_index = (df_sig['ngram_len'] == check_length)
    bigger_index = (df_sig['ngram_len'] == (check_length + 1))
    for ngram in df_sig[len_index]['ngram'].unique().tolist():
        ngram_index = (df_sig['ngram'] == ngram) & len_index
        ngram_count = df_sig[ngram_index]['ngram_count_All Reponses (Unfiltered)'].iloc[0]
        bigger_ngram_index = (
            bigger_index
            & (df_sig['ngram_count_All Reponses (Unfiltered)'] == ngram_count)
            & (df_sig['ngram'].str.startswith(ngram) | df_sig['ngram'].str.endswith(ngram))
        )
        if len(df_sig[bigger_ngram_index].index) > 0:
            print(f'Ngram {ngram} with count: {ngram_count} can be dropped')
            drops.append(ngram)
df_sig = df_sig[~df_sig['ngram'].isin(drops)]

Ngram t know with count: 13 can be dropped
Ngram t think with count: 9 can be dropped
Ngram t work with count: 6 can be dropped
Ngram irb tribal with count: 6 can be dropped
Ngram am working with count: 6 can be dropped
Ngram peoples or with count: 6 can be dropped
Ngram 12 24 with count: 5 can be dropped
Ngram 24 months with count: 5 can be dropped
Ngram at present with count: 5 can be dropped
Ngram in next with count: 5 can be dropped
Ngram next 12 with count: 5 can be dropped
Ngram present likely with count: 5 can be dropped
Ngram to change with count: 5 can be dropped
Ngram research work with count: 4 can be dropped
Ngram g https with count: 5 can be dropped
Ngram don t think with count: 9 can be dropped
Ngram don t work with count: 6 can be dropped
Ngram t work with with count: 6 can be dropped
Ngram indigenous peoples or with count: 6 can be dropped
Ngram peoples or descendant with count: 6 can be dropped
Ngram 12 24 months with count: 5 can be dropped
Ngram at present likely wit

In [15]:
df_sig.to_csv(ngram_chi_significant_path, index=False)
print(f'Saved significant ngrams to {ngram_chi_significant_path}')
display(df_sig.head(20))


Saved significant ngrams to /home/ekansa/oc-data/fair-care-ngram-freq-chi-tests-sig.csv


,Section,Sub-Section,ngram_len,fewer_Academic (Univ),fewer_Not Academic (Univ),fewer_CRM,fewer_N. America Focus,fewer_Outside N. America Focus,fewer_Responding as Individual,fewer_Responding as Organization,...,ngram_count_Outside N. America Focus,ngram_rate_over_all::Outside N. America Focus,ngram_count_Responding as Individual,ngram_rate_over_all::Responding as Individual,ngram_count_Responding as Organization,ngram_rate_over_all::Responding as Organization,chi_p_value::Academic (Univ)::vs::Not Academic (Univ),chi_p_value::CRM::vs::Academic (Univ),chi_p_value::Outside N. America Focus::vs::N. America Focus,chi_p_value::Responding as Organization::vs::Responding as Individual
5959,CARE,Authority to Control,2,,yes,yes,yes,,,,...,10.0,77.932569,11.0,-5.047335,8.0,-2.327558,0.020357,0.039578,0.006190,1.000000
6033,CARE,Authority to Control,2,,yes,yes,,,,,...,8.0,58.231532,13.0,11.643819,6.0,-30.847701,0.002054,0.048375,0.097157,0.521699
2431,CARE,Authority to Control,2,,yes,,yes,,,,...,10.0,86.681178,15.0,36.103136,3.0,-84.276643,0.002275,0.147165,0.001708,0.057870
2595,CARE,Authority to Control,2,,,,yes,,,,...,7.0,77.932569,8.0,-1.226270,6.0,4.570828,0.202803,0.664287,0.028316,1.000000
6021,CARE,Authority to Control,2,,yes,,yes,,,,...,9.0,98.151805,11.0,30.382090,3.0,-62.572519,0.006735,0.375706,0.000498,0.211302
8379,CARE,Authority to Control,3,,,,,,,yes,...,1.0,-96.269534,12.0,45.010470,1.0,-136.126560,0.768030,1.000000,0.356541,0.030964
3187,CARE,Authority to Control,2,,yes,,yes,,,yes,...,10.0,111.168126,12.0,45.898770,1.0,-136.733468,0.000091,0.354875,0.000009,0.028489
3351,CARE,Authority to Control,2,,yes,,yes,,,,...,10.0,111.168126,13.0,53.408493,NaN,NaN,0.000091,0.354875,0.000009,NaN
5481,CARE,Authority to Control,2,,yes,,,,,,...,4.0,33.413524,11.0,37.578071,2.0,-90.747071,0.035024,0.126496,0.676446,0.103730
6043,CARE,Authority to Control,2,,yes,,,,,,...,NaN,NaN,7.0,0.835638,5.0,1.754491,0.017011,NaN,NaN,1.000000


In [16]:
interesting_ngrams = [
    'academic unit',
    'as appropriate',
    'at present likely to change',
    'conference registration',
    'descendant communities',
    'don t know',
    'don t work with indigenous',
    'federal agency',
    'federal and',
    'google docs',
    'google sheets',
    'human remains',
    'i am working',
    'i don t know',
    'i don t think',
    'in place',
    'in publications',
    'indigenous communities',
    'indigenous data',
    'indigenous descendant',
    'indigenous people',
    'indigenous peoples',
    'institutional repository',
    'irb tribal review',
    'local communities',
    'made available',
    'my institution',
    'my research',
    'not applicable',
    'not applicable to my research',
    'not sure',
    'publicly accessible',
    'related to indigenous',
    'related to indigenous peoples',
    'some data',
    'state database',
    'submitted to',
    'the tribe',
    'tribal communities',
    'tribal lands',
    'university library',
    'we follow',
    'with indigenous data',
]


In [17]:

interest_index = df_sig['ngram'].isin(interesting_ngrams)
df_sig[interest_index].to_csv(ngram_chi_interesting_significant_path, index=False)